# Recommendation System Playground

Interactive evaluation notebook for the rec-system.
Run all cells, select a persona, choose versions, and compare feed quality.

**Guards:** Cells that require a live DB are guarded by `REC_PLAYGROUND_LIVE=true`.

In [ ]:
# Cell 1: Setup & imports
import os
import sys
from pathlib import Path

# Ensure project root is on sys.path
_ROOT = Path("__file__").parent.parent.resolve()
_NOTEBOOKS = _ROOT / "notebooks"
for _p in [str(_ROOT), str(_NOTEBOOKS)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Core helpers (always importable, no live DB required)
from notebooks.rec_playground_helpers import (
    color_code_position,
    render_diff_matrix,
    render_explainability_bars,
    render_llm_panel,
    render_metrics_table,
    render_side_by_side,
)

_LIVE = os.getenv("REC_PLAYGROUND_LIVE", "").lower() in ("1", "true", "yes")
print(f"Helpers loaded. Live mode: {_LIVE}")
print(f"Set REC_PLAYGROUND_LIVE=true to enable live DB calls.")

In [ ]:
# Cell 2: Load persona library
import os

if not os.getenv("REC_PLAYGROUND_LIVE"):
    print("Set REC_PLAYGROUND_LIVE=true to run live simulation")
else:
    from pathlib import Path
    from tests.evaluation.persona_loader import PersonaLoader

    _PERSONAS_DIR = Path("__file__").parent.parent / "tests" / "evaluation" / "personas"
    _PERSONAS_DIR = _PERSONAS_DIR.resolve()
    _loader = PersonaLoader(_PERSONAS_DIR)
    _personas = _loader.load_all()
    _persona_map = {p.id: p for p in _personas}
    print(f"Loaded {len(_personas)} personas: {[p.id for p in _personas]}")

In [ ]:
# Cell 3: ipywidgets — persona dropdown, version multi-select, runs counter, LLM checkbox
import os

if not os.getenv("REC_PLAYGROUND_LIVE"):
    print("Set REC_PLAYGROUND_LIVE=true to run live simulation")
else:
    import ipywidgets as widgets
    from IPython.display import display

    _persona_ids = [p.id for p in _personas]

    w_persona = widgets.Dropdown(
        options=_persona_ids,
        description="Persona:",
        layout=widgets.Layout(width="300px"),
    )

    w_versions = widgets.SelectMultiple(
        options=["baseline", "live_profile"],
        value=["baseline"],
        description="Versions:",
        layout=widgets.Layout(width="300px", height="80px"),
    )

    w_runs = widgets.IntSlider(
        value=1,
        min=1,
        max=5,
        step=1,
        description="Runs:",
        layout=widgets.Layout(width="300px"),
    )

    w_llm = widgets.Checkbox(
        value=False,
        description="Enable LLM Judge",
    )

    w_run_btn = widgets.Button(
        description="Run Benchmark",
        button_style="primary",
        layout=widgets.Layout(width="200px"),
    )

    display(w_persona, w_versions, w_runs, w_llm, w_run_btn)
    print("Widgets ready. Click 'Run Benchmark' to start.")

In [ ]:
# Cell 4: Handler that calls run_for_widgets() and renders results
import os

if not os.getenv("REC_PLAYGROUND_LIVE"):
    print("Set REC_PLAYGROUND_LIVE=true to run live simulation")
else:
    import asyncio
    from IPython.display import HTML, display

    _output_area = widgets.Output()
    display(_output_area)

    async def _run_benchmark_async(persona_id, versions, n_runs, use_llm):
        """Async core: run BenchmarkRunner for the chosen persona/versions."""
        from tests.evaluation.benchmark_runner import BenchmarkRunner, load_version_configs
        # NOTE: BenchmarkRunner requires live DB + session_factory.
        # Wire them from your environment or use the conftest factories.
        print(f"Would run: persona={persona_id}, versions={versions}, runs={n_runs}, llm={use_llm}")
        print("(Full live wiring requires DB connection — see conftest.py for fixture setup)")

    def _on_run_clicked(btn):
        with _output_area:
            _output_area.clear_output()
            persona_id = w_persona.value
            versions = list(w_versions.value)
            n_runs = w_runs.value
            use_llm = w_llm.value
            print(f"Running benchmark for persona={persona_id}, versions={versions}, runs={n_runs}")
            try:
                loop = asyncio.get_event_loop()
                loop.run_until_complete(
                    _run_benchmark_async(persona_id, versions, n_runs, use_llm)
                )
            except RuntimeError:
                # Jupyter has a running event loop
                import nest_asyncio
                nest_asyncio.apply()
                asyncio.get_event_loop().run_until_complete(
                    _run_benchmark_async(persona_id, versions, n_runs, use_llm)
                )

    w_run_btn.on_click(_on_run_clicked)
    print("Handler registered.")

In [ ]:
# Cell 5: Explainability demo — calls POST /recommendations/explain, renders bars
import os

if not os.getenv("REC_PLAYGROUND_LIVE"):
    print("Set REC_PLAYGROUND_LIVE=true to run live simulation")
else:
    import httpx
    from IPython.display import HTML, display

    _BASE_URL = os.getenv("REC_BASE_URL", "http://localhost:8000")
    _DEMO_USER_ID = os.getenv("REC_DEMO_USER_ID", "")
    _DEMO_CONTENT_ID = os.getenv("REC_DEMO_CONTENT_ID", "")

    if not _DEMO_USER_ID or not _DEMO_CONTENT_ID:
        print("Set REC_DEMO_USER_ID and REC_DEMO_CONTENT_ID env vars for explainability demo.")
    else:
        resp = httpx.post(
            f"{_BASE_URL}/recommendations/explain",
            json={"user_id": _DEMO_USER_ID, "content_id": _DEMO_CONTENT_ID},
            timeout=30,
        )
        resp.raise_for_status()
        payload = resp.json()

        # Build breakdown dict from response
        breakdown = {
            comp["name"]: {
                "value": comp.get("value"),
                "weight": comp.get("weight"),
                "contribution": comp.get("contribution"),
            }
            for comp in payload.get("components", [])
        }

        html_bars = render_explainability_bars(breakdown)
        display(HTML(f"<h3>Explain: user={_DEMO_USER_ID}, content={_DEMO_CONTENT_ID}</h3>{html_bars}"))

In [ ]:
# Cell 6: Persona evolution trace — topic_vector drift as line chart
import os

if not os.getenv("REC_PLAYGROUND_LIVE"):
    print("Set REC_PLAYGROUND_LIVE=true to run live simulation")
else:
    try:
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("matplotlib not available. Install with: uv add --dev matplotlib")
    else:
        # Example: simulate topic_vector snapshots over rounds
        # In real usage, capture profile.topic_vector after each BehaviorSimulator round.
        _topics = ["технологии", "наука", "бизнес", "политика", "спорт"]
        _rounds = list(range(1, 6))

        # Placeholder data — replace with real snapshots from BenchmarkRunner
        _rng = np.random.default_rng(42)
        _snapshots = _rng.dirichlet(np.ones(len(_topics)), size=len(_rounds))

        fig, ax = plt.subplots(figsize=(10, 4))
        for j, topic in enumerate(_topics):
            ax.plot(_rounds, _snapshots[:, j], marker="o", label=topic)
        ax.set_title("Topic Vector Drift Over Simulation Rounds (placeholder)")
        ax.set_xlabel("Round")
        ax.set_ylabel("Topic Weight")
        ax.legend(loc="upper right", fontsize=8)
        plt.tight_layout()
        plt.show()
        print("Drift chart shown (placeholder data). Wire real snapshots from BehaviorSimulator.")

## How to Use This Notebook

### Quick start (offline, no DB)

All cells are guarded with `REC_PLAYGROUND_LIVE`. Without the env var, each cell prints
a reminder and exits cleanly — no errors, no DB calls.

### Live mode

```bash
export REC_PLAYGROUND_LIVE=true
export REC_BASE_URL=http://localhost:8000      # for explainability demo
export REC_DEMO_USER_ID=<uuid>                # for explainability demo
export REC_DEMO_CONTENT_ID=<uuid>             # for explainability demo
uv run jupyter lab notebooks/rec_playground.ipynb
```

### Sections

| Cell | Purpose |
|------|---------|
| 1 | Setup — adds project root to sys.path, imports helpers |
| 2 | Load all 20 personas from `tests/evaluation/personas/` |
| 3 | Widget panel: persona dropdown, version selector, run counter, LLM toggle |
| 4 | Run handler: triggers benchmark, renders side-by-side + metrics table |
| 5 | Explainability: calls POST /recommendations/explain, renders bar chart |
| 6 | Drift trace: plots topic_vector evolution over simulation rounds |

### Helper functions

All logic lives in `notebooks/rec_playground_helpers.py` (importable without Jupyter):

- `color_code_position(pos)` — green/yellow/white CSS color by rank
- `render_side_by_side(persona, results)` — HTML table per version
- `render_metrics_table(results)` — ProxyMetrics HTML table
- `render_diff_matrix(results)` — rank changes between two versions
- `render_llm_panel(results)` — LLM judge summary table
- `render_explainability_bars(breakdown)` — inline SVG bar chart

### Running unit tests

```bash
uv run pytest tests/notebooks/test_rec_playground_helpers.py -v --no-cov
```

### Running the notebook smoke test

```bash
uv run pytest tests/notebooks/test_notebook_smoke.py -m slow --no-cov
```